# Execute metagenome taxonomic profile
Jacobo de la Cuesta-Zuluaga. August 2026.

The aim of this notebook is to obtain the taxonomic profile of metagenome samples.
For this, we'll use the `nf-core` pipeline [`taxprofiler`](https://nf-co.re/taxprofiler). 

## Before we start

The present notebook will continue using the sequence files we used in the `Sequence_QC`
notebook. Run notebook 01 first and wait until the QC job has finished. 
You should have `merged.R*.fastq.gz` files in `data/detaxizer/concatenated`.
Alternatively, you can use the output of detaxizer directly. These `filtered.fastq.gz`
files are in `detaxizer/filter/filtered`, and a sample sheet should be available in
`detaxizer/downstream_samplesheets`.

This notebook requires `conda` and the `Nextflow` and `VScode` environments of this repo.
Instructions to install `conda` are [here](https://conda.io/projects/conda/en/latest/user-guide/install/index.html).
If you are on the M3 cluster you should have conda available and if you executed the
`Sequence_QC` notebook, you should have the environments already configured. 


You can install the required environments once with:

```bash
cd Path/To/Metemgee
conda env create -f envs/Nextflow.yaml
conda env create -f envs/VScode.yaml
```
The notebooks are written in **R**, not Python. In VSCode, click the kernel selector on the top right
and pick the R kernel from the `VScode` environment.

## What you'll need to change

These are the only values you have to edit. Everything else can stay as it is.

| Variable | Where | What to put there |
|---|---|---|
| `base_dir` | Load libraries and set paths | The same one you used in notebook 01 |
| `repo_dir` | Load libraries and set paths | Where you cloned this repository |
| `seq_dir`, `detax_sheets_dir` | Load libraries and set paths | Only if your reads were not processed with `detaxizer` |
| `dbs_df` | Prepare databases | Only if you use different profilers or databases |
| `;-r 150` | Prepare databases | The read length of your data, only if using reads shorter than 150 bp |

## Load libraries and set paths

First, we'll set up the libraries and the work directory where we'll save our files.

In [2]:
# Libraries
library(tidyverse)
library(conflicted)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the ]8;;http://conflicted.r-lib.org/conflicted package]8;; to force all conflicts to become errors


In [3]:
# Housekeeping: tells R which `filter` function to use when more than one
# package provides one. Nothing to change here.
conflicts_prefer(dplyr::filter)

[conflicted] Will prefer dplyr::filter over any other package.


The following chunk will define the directories where the data is stored and where the output will be
saved. The present example assumes everything will be contained in the same directory: `base_dir`.
This might be different in your particular case, for example, if your sequences are stored on a
centralized directory or you have multiple runs stored in different folders. You can change this
accordingly.

`base_dir` has to exist already; in this case it is the same we defined in the
`Sequence_QC` notebook.

If a folder already exists, `dir.create` prints a warning. That's harmless.

The configuration file is a different matter: it lives in the cloned repository, which is not
necessarily inside `base_dir`.

In [ ]:
# Directories
# Base directory
base_dir <-"/PATH/TO/YOUR/FOLDER"

# Where you cloned the Metemgee repository.
repo_dir <- "/PATH/TO/YOUR/REPO"

# Data
data_dir <- file.path(base_dir, "data")
dir.create(data_dir)

# Output dir
taxprofiler_dir <- file.path(data_dir, "taxprofiler")
dir.create(taxprofiler_dir)

# Nextflow intermediate files. Can be deleted once the run is done
work_dir <- file.path(data_dir, "nextflow_work")
dir.create(work_dir)

# Sheets dir
sheets_dir <- file.path(data_dir, "sheets")
dir.create(sheets_dir)

# Detaxizer outputs
# Change if using sequences processed elsewhere
seq_dir <- file.path(data_dir, "detaxizer/concatenated")

# Nextflow intermediate files. Can be deleted once the run is done
nextflow_dir <- file.path(data_dir, "nextflow_work")
dir.create(nextflow_dir)

# Software
conda_env <- "Nextflow"
config_file <- file.path(repo_dir, "config/taxprofiler.config")

Warning message:
In dir.create(data_dir) :
  '/mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data' already exists
Warning message:
In dir.create(taxprofiler_dir) :
  '/mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/taxprofiler' already exists
Warning message:
In dir.create(work_dir) :
  '/mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/nextflow_work' already exists
Warning message:
In dir.create(sheets_dir) :
  '/mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/sheets' already exists
Warning message:
In dir.create(nextflow_dir) :
  '/mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/nextflow_work' already exists


In [5]:
# Check that the base and sequence directories are defined and exist
stopifnot(dir.exists(base_dir), dir.exists(seq_dir))

## Samples file

As mentioned above, we will use the samples file generated by `detaxizer`. If you
are using samples pre-processed in a different manner or you want to use the
built-in processing from the `taxprofiler` pipeline, you will need to generate
the samples file yourself. You can generate this file manually or programmatically,
as we did in the last notebook. You can read a detailed description of this table
on the `taxprofiler` documentation page [here](https://nf-co.re/taxprofiler/2.0.1/docs/usage/).

Briefly, you need a `csv` table with the following fields:

| Column | What goes in it |
|---|---|
| `sample` | Sample name. Identical across rows for several runs of the same sample |
| `run_accession` | Name of the individual sequencing run |
| `instrument_platform` | The sequencing technology, `ILLUMINA` for short reads |
| `fastq_1`, `fastq_2` | Full path of the forward and reverse files |
| `fasta` | Left empty when using `fastq` files |

If you have multiple `detaxizer` runs, you can load and combine the tables to
perform a single `taxprofiler` run.

### A note on multiple sequencing runs

A single sample is often sequenced more than once, either because it was split across
several lanes, or because a first run didn't reach the desired sequencing depth. Each of
those sequencing pairs counts as a separate run, and each one gets its own row in the
samples file.

Two columns control this. The `sample` identifier has to be the same across all 
rows that come from the same biological sample, while each `run_accession` has to
be unique between runs of the same sample (for example, `Sample_1`, could have
`run_1`, and `run_2`, but not `run_1` twice).  

With `--perform_runmerging`, the pipeline concatenates the `fastq` files of the runs
that share a sample name before profiling. This option is similar to the concatenation
step performed in the `Sequence_QC` notebook. Without that argument, each row is profiled
on its own and you end up with one profile per run instead of one per sample.

In the present example, we'll use the reads we concatenated in the `Sequence_QC` 
notebook, therefore, the reads from multiple runs are already combined. We'll
create the samples file programatically. You can also do this manually on Excel.

In [6]:
# List clean sequences
clean_seq_list <- list.files(seq_dir, pattern = "fastq.gz", full.names = TRUE)

# Forward reads
forward_reads <- clean_seq_list |> 
  str_subset("R1")
# Reverse reads
reverse_reads <- clean_seq_list |> 
  str_subset("R2")

In [7]:
# Create a single data frame for taxprofiler
samples_table <- data.frame(
  fastq_1 = forward_reads, 
  fastq_2 = reverse_reads, 
  instrument_platform = "ILLUMINA", 
  fasta = ""
) |> 
  mutate(
    sample = basename(fastq_1), 
    sample = str_remove(sample, "_merged.*")
  ) |> 
  group_by(sample) |> 
  mutate(run_accession = str_c("run_", row_number())) |> 
  ungroup() |> 
  relocate(sample, instrument_platform, run_accession)

samples_table |> 
  head()

# A tibble: 2 × 6
  sample   instrument_platform run_accession fastq_1               fastq_2 fasta
  <chr>    <chr>               <chr>         <chr>                 <chr>   <chr>
1 MI-142-H ILLUMINA            run_1         /mnt/lustre/groups/m… /mnt/l… ""   
2 MI-237-H ILLUMINA            run_1         /mnt/lustre/groups/m… /mnt/l… ""   

In [8]:
# Write samples file
taxprofiler_samplesfile = file.path(sheets_dir, "Example_taxprofiler_samples.csv")
write_csv(samples_table, file = taxprofiler_samplesfile)

For instructions on how to create the samples table if your sequences are stored
in multiple locations, see the `Sequence_QC` notebook.

## Prepare databases for pipeline execution

The pipeline requires a file with the location of the databases for the software to be used. 

In the present example we'll use `kraken`/`Bracken` for taxonomic profiling, combined
with databases derived from the Unified Human Gastrointestinal Genome (UHGG) catalog
available [here](https://ftp.ebi.ac.uk/pub/databases/metagenomics/mgnify_genomes/human-gut/v2.0.2/). 
This is derived from human metagenomes and isolates, therefore, it gives a good coverage
for this kind of sample, but it might be poor for other sources. For a catalog of mouse gut microbes, 
see [here](https://ftp.ebi.ac.uk/pub/databases/metagenomics/mgnify_genomes/mouse-gut/v1.0/)

For this example you don't need to download the UHGG catalog if you're using the 
centralized database folder of the Maier Lab. The `kraken` and `Bracken` files are
located in the same directory, that's why the same path is used for both in the 
chunk below. 

For instructions to use other profilers and download their respective databases
see [here](https://nf-co.re/taxprofiler/2.0.1/docs/usage/).

In [ ]:
# Taxdump folder
taxdump_dir = "/mnt/lustre/groups/maier/databases/Kraken_Bracken/k2_uhgg/taxonomy"

# Create dbs file
dbs_df = data.frame(tool = c("kraken2","bracken"),
    db_name = c("k2_uhgg", "B_uhgg"),
    db_params = c("", ";-r 150"),
    db_path = c("/mnt/lustre/groups/maier/databases/Kraken_Bracken/k2_uhgg/k2_uhgg_v2.0.2.tar.gz",
                "/mnt/lustre/groups/maier/databases/Kraken_Bracken/k2_uhgg/k2_uhgg_v2.0.2.tar.gz")) 

# Print head
dbs_df |> 
  head()

     tool db_name db_params
1 kraken2 k2_uhgg          
2 bracken  B_uhgg   ;-r 150
                                                                          db_path
1 /mnt/lustre/groups/maier/databases/Kraken_Bracken/k2_uhgg/k2_uhgg_v2.0.2.tar.gz
2 /mnt/lustre/groups/maier/databases/Kraken_Bracken/k2_uhgg/k2_uhgg_v2.0.2.tar.gz

In [10]:
# Write file
dbs_file = file.path(sheets_dir, "Taxprofiler_databases_file.csv")
dbs_df |> 
    write_csv(dbs_file)

## Execute pipeline

The code below constructs the bash command to activate the conda environment, change to the output
directory, and run the `taxprofiler` pipeline with all required arguments and resources.

What the arguments do: 

- `--input` and `--databases`: the samples file and the database file built above
- `--perform_shortread_redundancyestimation`: runs `nonpareil` to estimate how completely the 
 metagenome was sequenced
- `--run_kraken2` and `--run_bracken`: the profilers to use. `kraken2` assigns reads to taxa,
 `bracken` re-estimates the abundances from those assignments
- `--run_profile_standardisation` and the `--taxpasta_*` arguments: merge the per-sample profiles into
 one table and add the taxon names, ranks and full lineage
- `--taxpasta_taxonomy_dir`: the taxonomy that those names are looked up in
- `-profile` and `-c`: cluster settings and resource allocation



**Note** that in the current execution of the pipeline, no read trimming and host
read removal are performed. This was done in the last `Sequence_QC` notebook. 
If you want to use the built-in steps from `taxprofiler`, you can use the arguments: `--perform_shortread_qc`, `--perform_shortread_hostremoval`, 
`--perform_shortread_complexityfilter`, `--shortread_qc_dedup`, 
`--shortread_qc_minlength [MINIMUM LENGTH]` and `--hostremoval_reference [HOST GENOME FILE]`.
You can find the explanation for these arguments [here](https://nf-co.re/taxprofiler/2.0.1/parameters/).

In [11]:
# Base command
taxprofiler_cmd = str_glue(
  "conda activate {{conda_env}} && \\
  cd {{out_dir}} && \\
  nextflow run nf-core/taxprofiler -r 2.0.1 \\
  --input {{samples_sheet}} \\
  --databases {{databases_sheet}} \\
  --outdir {{out_dir}} \\
  -c {{config_file}} \\
  -work-dir {{nextflow_dir}} \\
  -profile m3c \\
  --perform_shortread_redundancyestimation \\
  --run_profile_standardisation \\
  --taxpasta_taxonomy_dir {{tax_dir}} \\
  --taxpasta_add_name \\
  --taxpasta_add_rank \\
  --taxpasta_add_lineage \\
  --taxpasta_add_ranklineage \\
  --run_kraken2 \\
  --run_bracken")

Now we can replace the placeholders in the `taxprofiler` command template with the actual paths and filenames
defined above. Then, the chunk prints the full command for you to copy and run in your terminal.

In [12]:
# Fill command
taxprofiler_filled = str_glue(taxprofiler_cmd,
                         conda_env = conda_env,
                         samples_sheet = taxprofiler_samplesfile,
                         databases_sheet = dbs_file,
                         tax_dir = taxdump_dir,
                         out_dir = taxprofiler_dir,
                         config_file = config_file)

taxprofiler_filled

conda activate Nextflow && cd /mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/taxprofiler && nextflow run nf-core/taxprofiler -r 2.0.1 --input /mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/sheets/Example_taxprofiler_samples.csv --databases /mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/sheets/Taxprofiler_databases_file.csv --outdir /mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/taxprofiler -c /mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/config/taxprofiler.config -work-dir /mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/nextflow_work -profile m3c --perform_shortread_redundancyestimation --run_profile_standardisation --taxpasta_taxonomy_dir /mnt/lustre/groups/maier/databases/Kraken_Bracken/k2_uhgg/taxonomy --taxpasta_add_name --taxpasta_add_rank --taxpasta_add_lineage --taxpasta_add_ranklineage --run_kr

## While it runs

The pipeline takes several hours and stops if your connection to the cluster drops. Start it inside a
`tmux` or `screen` session so it keeps running when you close the terminal.

Nextflow prints one line per step. If a step fails, adding `-resume` to the command restarts the run
from that point instead of from the beginning.

## What you should have at the end

Inside `data/taxprofiler`:

- the `multiqc` report, which is the one to open first
- a calculation of metagenome coverage in the `taxprofiler/nonpareil` directory.
 The `nonpareil_all_samples.tsv` file aggregates the results of all samples. 
 See the `nonpareil` documentation [here](https://nonpareil.readthedocs.io/en/latest/redundancy.html)
- tables with species abundance on each of the samples :`taxprofiler/taxpasta/bracken_B_uhgg.tsv`
 and `taxprofiler/bracken/bracken_B_uhgg_combined_reports.txt` in the output 
 folder you specified. Both results are very similar, the difference is that
  `taxpasta` includes the complete taxonomic classification of each microbe 
  found, not only the species name or ID.

Once you are satisfied with the results, `data/nextflow_work` can be deleted.